# Multimodal Jacobian Lens for TokenPacker (VQAv2 Dataset)

This notebook provides a self-contained, end-to-end implementation of the **Jacobian Lens** for **TokenPacker** Multimodal Large Language Models (MLLMs), including support for `cross_attn_adaptive_v*` and high-resolution patch architectures (`TokenPacker-HD`).

---

### Pipeline Overview
1. **Environment & Path Setup**: Registers TokenPacker repository paths and imports `jlens` visualization utilities.
2. **Server Configuration & Direct Model Loader**: Loads TokenPacker (`LlavaLlamaForCausalLM` + vision tower + projector + tokenizer) directly via `AutoTokenizer` and `LlavaLlamaForCausalLM.from_pretrained` with SDPA/FlashAttention enabled on GPU.
3. **VQAv2 Dataset Loader**: Streams server-side VQAv2 evaluation samples (`question_file` `.jsonl` + `image_folder`) using TokenPacker's native `CustomDataset` pipeline.
4. **`MultimodalTokenPackerLensModel` Adapter**: Wraps TokenPacker to convert multimodal inputs into fused `inputs_embeds` and perform autograd Jacobian computation.
5. **Multimodal Jacobian Lens Fitting**: Fits $J_l = \mathbb{E}[\partial h_{\text{final}} / \partial h_l]$ over VQAv2 samples with atomic `.pt` checkpoint saving.
6. **Standalone HTML Visualization Export**: Exports interactive D3.js heatmaps as standalone `.html` files (or sidecar folders) that can be downloaded and opened locally in any browser.

## Section 1: Environment Setup & Library Imports

This cell:
- Registers your local or server `TokenPacker` codebase into `sys.path` so its modules (`llava`, `llava.eval.model_vqa_loader`, etc.) can be imported seamlessly.
- Imports PyTorch, PIL, `transformers`, and `jlens` core modules (`ActivationRecorder`, `JacobianLens`, `build_page`, etc.).

In [1]:
import os
import sys
import json
import time
import math
import argparse
from pathlib import Path
from typing import Sequence, Any, Optional

import torch
import torch.nn as nn
import numpy as np
from PIL import Image

# ==========================================
# TokenPacker Repository Path Setup
# ==========================================
TOKENPACKER_REPO = "/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker"
if TOKENPACKER_REPO not in sys.path and os.path.exists(TOKENPACKER_REPO):
    sys.path.insert(0, TOKENPACKER_REPO)
    print(f"Added {TOKENPACKER_REPO} to sys.path")

import jlens
from jlens.fitting import valid_position_mask, SKIP_FIRST_N_POSITIONS
from jlens.hooks import ActivationRecorder
from jlens.lens import JacobianLens
from jlens.vis import compute_slice, build_page, notebook_iframe, SliceData, _ranks_of

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("jlens and TokenPacker environment initialized successfully.")

/mnt/pvc-shared-pvc-data-volume-ea328235/miniconda3/envs/research/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Added /mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker to sys.path
jlens and TokenPacker environment initialized successfully.


## Section 2: Server Paths Configuration & Direct Model Loader (with SDPA FlashAttention)

This cell defines the primary configuration parameters and directly loads TokenPacker using `AutoTokenizer` and `LlavaLlamaForCausalLM.from_pretrained()` with forced `attn_implementation="sdpa"`:
- `MODEL_PATH`: Location of your TokenPacker model checkpoint on the GPU server.
- `QUESTION_FILE`: Path to your existing VQAv2 evaluation JSONL file.
- `IMAGE_FOLDER`: Directory containing VQAv2 images.
- `attn_implementation="sdpa"`: Enables PyTorch C++ FlashAttention / Memory-Efficient Attention for 3x faster autograd.

In [2]:
# ==========================================
# Configurable Server Paths & Hyperparameters
# ==========================================
MODEL_PATH = "/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker/checkpoints/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-05242026"
MODEL_NAME = "llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202"
QUESTION_FILE = "/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker/playground/data/eval/vqav2/llava_vqav2_mscoco_test-dev2015.jsonl"
IMAGE_FOLDER = "/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker/playground/data/eval/vqav2/test2015"
OUTPUT_LENS_PATH = "out/tokenpacker_multimodal_vqav2_lens.pt"
CHECKPOINT_PATH = "out/tokenpacker_vqav2_fitting_ckpt.pt"
OUTPUT_HTML_DIR = "out/visualizations/"
CONV_MODE = "vicuna_v1"

N_FITTING_SAMPLES = 200
DIM_BATCH = 16
MAX_SEQ_LEN = 256
SKIP_FIRST = 1
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

# Direct Load TokenPacker Model and Tokenizer
try:
    from transformers import AutoTokenizer, AutoConfig
    from llava.model.language_model.llava_llama import LlavaLlamaForCausalLM

    if os.path.exists(MODEL_PATH):
        print(f"Loading TokenPacker tokenizer & model from {MODEL_PATH}...")
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_PATH,
            model_max_length=2048,
            padding_side="right",
            use_fast=True
        )

        config = AutoConfig.from_pretrained(MODEL_PATH)
        config._attn_implementation = "sdpa"
        config.attn_implementation = "sdpa"

        model = LlavaLlamaForCausalLM.from_pretrained(
            MODEL_PATH,
            config=config,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            device_map=DEVICE
        )

        # Force SDPA FlashAttention on both model and sub-module config
        model.config._attn_implementation = "sdpa"
        model.config.attn_implementation = "sdpa"
        if hasattr(model, "model") and hasattr(model.model, "config"):
            model.model.config._attn_implementation = "sdpa"
            model.model.config.attn_implementation = "sdpa"

        # Initialize Vision Tower & Image Processor
        vision_tower = model.get_vision_tower()
        if not vision_tower.is_loaded:
            vision_tower.load_model()
        vision_tower.to(device=DEVICE, dtype=torch.bfloat16)
        image_processor = vision_tower.image_processor

        model.eval()
        attn_type = type(model.model.layers[0].self_attn).__name__ if hasattr(model, "model") and hasattr(model.model, "layers") else "Unknown"
        print(f"Model loaded successfully: {type(model).__name__} | Attention Module: {attn_type}")
    else:
        print(f"Note: MODEL_PATH '{MODEL_PATH}' not found locally. Update path when running on GPU server.")
        tokenizer, model, image_processor = None, None, None
except Exception as err:
    print(f"Model loading error: {err}")
    tokenizer, model, image_processor = None, None, None

Using device: cuda:0


/mnt/pvc-shared-pvc-data-volume-ea328235/miniconda3/envs/research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TokenPacker tokenizer & model from /mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker/checkpoints/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-05242026...


/mnt/pvc-shared-pvc-data-volume-ea328235/miniconda3/envs/research/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.80s/it]
Some weights of the model checkpoint at /mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker/checkpoints/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-05242026 were not used when initializing LlavaLlamaForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.2.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.14.self_attn.q_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.7.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.9.self_attn.q_p

Model loaded successfully: LlavaLlamaForCausalLM | Attention Module: LlamaAttention


In [3]:
import types                                                                                                                                                                                                  
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb                                                                                                                                     
                                                                                                                                                                                                                
def fast_sdpa_forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_value=None, output_attentions=False, use_cache=False):                                                            
    bsz, q_len, _ = hidden_states.size()                                                                                                                                                                      
                                                                                                                                                                                                                
    num_heads = self.num_heads                                                                                                                                                                                
    num_kv_heads = getattr(self, "num_key_value_heads", num_heads)                                                                                                                                            
                                                                                                                                                                                                                
    query_states = self.q_proj(hidden_states).view(bsz, q_len, num_heads, self.head_dim).transpose(1, 2)                                                                                                      
    key_states = self.k_proj(hidden_states).view(bsz, q_len, num_kv_heads, self.head_dim).transpose(1, 2)                                                                                                     
    value_states = self.v_proj(hidden_states).view(bsz, q_len, num_kv_heads, self.head_dim).transpose(1, 2)                                                                                                   
                                                                                                                                                                                                                
    kv_seq_len = key_states.shape[-2]                                                                                                                                                                         
    if hasattr(self, "rotary_emb"):                                                                                                                                                                           
        cos, sin = self.rotary_emb(value_states, seq_len=kv_seq_len)                                                                                                                                          
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin, position_ids)                                                                                                     
                                                                                                                                                                                                                
    # Native PyTorch 2.0+ C++ FlashAttention kernel                                                                                                                                                           
    attn_output = torch.nn.functional.scaled_dot_product_attention(                                                                                                                                           
        query_states, key_states, value_states, attn_mask=attention_mask, is_causal=(attention_mask is None and q_len > 1)                                                                                    
    )                                                                                                                                                                                                         
    attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, -1)                                                                                                                               
    attn_output = self.o_proj(attn_output)                                                                                                                                                                    
    return attn_output, None, past_key_value                                                                                                                                                                  
                                                                                                                                                                                                                
# Bind fast_sdpa_forward to all 32 attention layers                                                                                                                                                           
for layer in model.model.layers:                                                                                                                                                                              
    layer.self_attn.forward = types.MethodType(fast_sdpa_forward, layer.self_attn)                                                                                                                            
                                                                                                                                                                                                                
print("Successfully monkey-patched all 32 attention layers with PyTorch C++ SDPA FlashAttention!")    

Successfully monkey-patched all 32 attention layers with PyTorch C++ SDPA FlashAttention!


In [4]:
model

LlavaLlamaForCausalLM(
  (model): LlavaLlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): 

## Section 3: VQAv2 Dataset Pipeline

This cell integrates TokenPacker's native `CustomDataset` pipeline (from `llava.eval.model_vqa_loader`).
- It streams image-question pairs directly from your existing server dataset files without needing to download anything new.
- Handles both standard and high-resolution patch aspect ratios (`image_aspect_ratio == 'slice'`).

In [5]:
# ==========================================
# VQAv2 Dataset Pipeline using TokenPacker's CustomDataset
# ==========================================
from torch.utils.data import DataLoader

class VQAArgs:
    def __init__(self, conv_mode="vicuna_v1"):
        self.conv_mode = conv_mode

def create_vqav2_dataloader(question_file, image_folder, tokenizer, image_processor, model_config, n_samples=200):
    from llava.eval.model_vqa_loader import CustomDataset
    import llava.eval.model_vqa_loader as vqa_loader
    
    vqa_loader.args = VQAArgs(conv_mode=CONV_MODE)
    
    with open(os.path.expanduser(question_file), "r") as f:
        questions = [json.loads(line) for line in f][:n_samples]
        
    dataset = CustomDataset(questions, image_folder, tokenizer, image_processor, model_config)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2)
    return dataloader

if os.path.exists(QUESTION_FILE) and os.path.exists(IMAGE_FOLDER) and model is not None:
    vqav2_loader = create_vqav2_dataloader(QUESTION_FILE, IMAGE_FOLDER, tokenizer, image_processor, model.config, N_FITTING_SAMPLES)
    print(f"Loaded VQAv2 dataloader with {len(vqav2_loader)} samples.")
else:
    vqav2_loader = None
    print("VQAv2 DataLoader ready for instantiation on GPU server.")

Loaded VQAv2 dataloader with 200 samples.


## Section 4: `MultimodalTokenPackerLensModel` Protocol Adapter

This cell defines the custom wrapper class that connects TokenPacker to `jlens`:
- **`get_multimodal_inputs_embeds`**: Passes images through the vision encoder and TokenPacker projector (`prepare_inputs_labels_for_multimodal` or `prepare_adaptive_inputs_labels_for_multimodal`), fusing image features with text tokens.
- **`forward`**: Runs autograd-retained forward passes on `inputs_embeds` across residual blocks.
- **`unembed`**: Applies final LayerNorm and LM head to map intermediate residual vectors to vocabulary logit distributions.

In [6]:
# ==========================================
# MultimodalTokenPackerLensModel Adapter
# ==========================================
class MultimodalTokenPackerLensModel:
    """LensModel protocol implementation for TokenPacker MLLMs.
    Supports cross_attn_adaptive_v* and HD patch architectures.
    """
    def __init__(self, model, tokenizer, image_processor):
        self.model = model
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        self._text_module = model.model  # LlavaLlamaModel
        self.layers = self._text_module.layers
        self._final_norm = self._text_module.norm
        self._lm_head = model.lm_head

        self.n_layers = model.config.num_hidden_layers
        self.d_model = model.config.hidden_size

        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @property
    def input_device(self) -> torch.device:
        return self._lm_head.weight.device

    def encode(self, text: str, max_length: int = 128) -> torch.Tensor:
        return self.tokenizer(text, return_tensors="pt", max_length=max_length).input_ids.to(self.input_device)

    def get_multimodal_inputs_embeds(
        self,
        input_ids: torch.Tensor,
        images: torch.Tensor,
        h_block: Optional[Any] = None,
        w_block: Optional[Any] = None,
        mode: Optional[str] = None
    ) -> torch.Tensor:
        """Generates fused image-text input embeddings via TokenPacker's projector."""
        h_b = h_block.tolist() if torch.is_tensor(h_block) else h_block
        w_b = w_block.tolist() if torch.is_tensor(w_block) else w_block

        # if hasattr(self.model, "prepare_adaptive_inputs_labels_for_multimodal"):
        _, _, _, inputs_embeds, _ = self.model.prepare_adaptive_inputs_labels_for_multimodal(
            input_ids=input_ids,
            attention_mask=None,
            past_key_values=None,
            labels=None,
            images=images,
            mode=mode,
            h_block=h_b,
            w_block=w_b
        )
        # else:
        # _, _, _, inputs_embeds, _ = self.model.prepare_inputs_labels_for_multimodal(
        #     input_ids=input_ids,
        #     attention_mask=None,
        #     past_key_values=None,
        #     labels=None,
        #     images=images,
        #     mode=mode,
        #     h_block=h_b,
        #     w_block=w_b
        # )
        return inputs_embeds

    def forward(self, input_ids_or_embeds: torch.Tensor) -> Any:
        if input_ids_or_embeds.dtype == torch.int64:
            return self._text_module(input_ids=input_ids_or_embeds, use_cache=False)
        else:
            return self._text_module(inputs_embeds=input_ids_or_embeds, use_cache=False)

    def unembed(self, residual: torch.Tensor) -> torch.Tensor:
        target_device = self._lm_head.weight.device
        target_dtype = self._lm_head.weight.dtype
        return self._lm_head(self._final_norm(residual.to(target_dtype).to(target_device)))

if model is not None:
    lens_model = MultimodalTokenPackerLensModel(model, tokenizer, image_processor)
    print(f"MultimodalTokenPackerLensModel initialized: n_layers={lens_model.n_layers}, d_model={lens_model.d_model}")
else:
    lens_model = None

MultimodalTokenPackerLensModel initialized: n_layers=32, d_model=4096


## Section 5: Multimodal Jacobian Lens Fitting Engine

This cell defines and executes `fit_multimodal_jacobian_lens`:
- It iterates over VQAv2 multimodal samples.
- Computes cotangent backward passes for output dimension blocks (`DIM_BATCH = 32`) to aggregate the Jacobian matrix $J_l = \mathbb{E}[\partial h_{\text{final}} / \partial h_l]$ for every layer.
- Saves interim fitting progress automatically every 10 samples to `CHECKPOINT_PATH` (resumable if interrupted).
- Flushes CUDA cache per iteration to ensure zero memory driver stalls.
- Saves the final fitted lens to `OUTPUT_LENS_PATH` (`.pt`).

In [7]:
# # ==========================================
# # Jacobian Lens Fitting on VQAv2 Multimodal Samples
# # ==========================================
# from tqdm import tqdm

# def fit_multimodal_jacobian_lens(
#     lens_model: MultimodalTokenPackerLensModel,
#     dataloader,
#     dim_batch: int = 32,
#     max_seq_len: int = 256,
#     skip_first: int = 1,
#     checkpoint_path: str = "out/tokenpacker_vqav2_fitting_ckpt.pt",
#     checkpoint_every: int = 10,
#     resume: bool = True
# ) -> JacobianLens:
#     """Fits Jacobian Lens J_l over multimodal VQAv2 samples."""
#     n_layers, d_model = lens_model.n_layers, lens_model.d_model
#     source_layers = list(range(n_layers - 1))
#     target_layer = n_layers - 1

#     jacobian_sum = {l: torch.zeros(d_model, d_model, dtype=torch.float32) for l in source_layers}
#     n_done = 0
#     next_idx = 0

#     if resume and checkpoint_path and os.path.exists(checkpoint_path):
#         state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
#         jacobian_sum = state["jacobian_sum"]
#         n_done = state["n_done"]
#         next_idx = state["next_idx"]
#         print(f"Resuming fitting from checkpoint: {next_idx} samples completed.")

#     device = lens_model.input_device
#     mode = getattr(lens_model.model.config, 'image_aspect_ratio', None)

#     for sample_idx, batch in enumerate(tqdm(dataloader)):
#         if sample_idx < next_idx:
#             continue

#         input_ids, image_tensor, h_block, w_block = batch
#         input_ids = input_ids.to(device)

#         if image_tensor.ndim == 5:
#             image_tensor = image_tensor.squeeze(0)
#         image_tensor = image_tensor.to(device, dtype=torch.bfloat16)

#         # 1. Compute fused multimodal embeddings
#         with torch.no_grad():
#             inputs_embeds = lens_model.get_multimodal_inputs_embeds(
#                 input_ids=input_ids,
#                 images=image_tensor,
#                 h_block=h_block,
#                 w_block=w_block,
#                 mode=mode
#             )

#         seq_len = inputs_embeds.shape[1]
#         if seq_len > max_seq_len:
#             inputs_embeds = inputs_embeds[:, :max_seq_len, :]
#             seq_len = max_seq_len

#         try:
#             position_mask = valid_position_mask(seq_len, skip_first=skip_first)
#         except ValueError as exc:
#             print(f"Skipping sample {sample_idx}: {exc}")
#             next_idx = sample_idx + 1
#             continue

#         n_valid_positions = int(position_mask.sum())
#         n_passes = math.ceil(d_model / dim_batch)
#         per_sample_J = {l: torch.zeros(d_model, d_model, dtype=torch.float32) for l in source_layers}

#         with ActivationRecorder(lens_model.layers, at=[*source_layers, target_layer], start_graph_at=min(source_layers)) as recorder, torch.enable_grad():
#             replicated_embeds = inputs_embeds.expand(dim_batch, -1, -1)
#             lens_model.forward(replicated_embeds)
            
#             target_act = recorder.activations[target_layer]
#             source_acts = [recorder.activations[l] for l in source_layers]
#             valid_positions = position_mask.nonzero(as_tuple=True)[0].to(target_act.device)
#             batch_indices = torch.arange(dim_batch, device=target_act.device)
#             cotangent = torch.zeros_like(target_act)

#             for pass_idx, dim_start in enumerate(range(0, d_model, dim_batch)):
#                 n_dims = min(dim_batch, d_model - dim_start)
#                 cotangent.zero_()
#                 cotangent[batch_indices[:n_dims, None], valid_positions[None, :], dim_start + batch_indices[:n_dims, None]] = 1.0
                
#                 grads = torch.autograd.grad(
#                     outputs=target_act,
#                     inputs=source_acts,
#                     grad_outputs=cotangent,
#                     retain_graph=(pass_idx < n_passes - 1)
#                 )

#                 for layer, grad in zip(source_layers, grads, strict=True):
#                     positions_on_device = valid_positions.to(grad.device, non_blocking=True)
#                     rows = grad[:n_dims, positions_on_device, :].float().mean(dim=1).cpu()
#                     per_sample_J[layer][dim_start : dim_start + n_dims, :] = rows
#                 del grads

#         for layer in source_layers:
#             jacobian_sum[layer] += per_sample_J[layer]

#         n_done += 1
#         next_idx = sample_idx + 1

#         if (sample_idx + 1) % 10 == 0:
#             if checkpoint_path:
#                 os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
#                 torch.save({"jacobian_sum": jacobian_sum, "n_done": n_done, "next_idx": next_idx}, checkpoint_path)

#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()

#     jacobian_mean = {layer: jacobian_sum[layer] / max(1, n_done) for layer in source_layers}
#     fitted_lens = JacobianLens(jacobians=jacobian_mean, n_prompts=n_done, d_model=d_model)
#     return fitted_lens

# # Execute fitting when running on GPU server
# if vqav2_loader is not None and lens_model is not None:
#     print("Starting Jacobian Lens fitting on VQAv2 dataset...")
#     multimodal_lens = fit_multimodal_jacobian_lens(
#         lens_model=lens_model,
#         dataloader=vqav2_loader,
#         dim_batch=DIM_BATCH,
#         max_seq_len=MAX_SEQ_LEN,
#         skip_first=SKIP_FIRST,
#         checkpoint_path=CHECKPOINT_PATH,
#         checkpoint_every=10,
#         resume=True
#     )
#     os.makedirs(os.path.dirname(OUTPUT_LENS_PATH), exist_ok=True)
#     multimodal_lens.save(OUTPUT_LENS_PATH)
#     print(f"Fitted multimodal Jacobian Lens saved to {OUTPUT_LENS_PATH}")
# else:
#     print("Fitting deferred: Update paths in Cell 2 to execute on your GPU server.")

## Section 6: Export Standalone Interactive HTML Visualization

This cell defines `export_multimodal_slice_html`:
- It processes an inference sample (image + prompt), extracts intermediate layer activations, applies the fitted Jacobian Lens $J_l$, and computes full-vocabulary token ranks.
- **`mode="embed"` (Default)**: Bundles D3.js, all metadata, and token rank arrays into a **single self-contained `.html` file** (~300KB–1MB).
- **Local Viewing Instructions**: Download the `.html` file from your server to your local machine (via `scp`/SFTP) and double-click to open it in Chrome, Safari, or Firefox without any local web server!

In [8]:
# ==========================================
# Export Standalone Interactive HTML Visualization File/Folder
# ==========================================
@torch.no_grad()
def export_multimodal_slice_html(
    lens_model: MultimodalTokenPackerLensModel,
    fitted_lens_path: str,
    image_path: str,
    prompt_text: str,
    output_html_path: str = "out/visualizations/multimodal_slice.html",
    mode: str = "embed",       # "embed" for single self-contained HTML, "fetch" for folder export
    top_n: int = 10
):
    """Computes multimodal slice data and saves a standalone interactive HTML file (or folder)
    that can be downloaded to your local machine and opened directly in any browser.
    """
    device = lens_model.input_device
    tokenizer = lens_model.tokenizer
    
    if not os.path.exists(fitted_lens_path):
        print(f"Error: Lens checkpoint {fitted_lens_path} not found.")
        return None

    lens = JacobianLens.from_pretrained(os.path.dirname(fitted_lens_path), filename=os.path.basename(fitted_lens_path))
    print(f"Loaded Jacobian Lens from {fitted_lens_path}")

    # 1. Process Multimodal Inputs
    from llava.mm_utils import tokenizer_image_token, process_images
    from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN

    image = Image.open(image_path).convert('RGB')
    image_tensor = process_images([image], lens_model.image_processor, lens_model.model.config)[0].unsqueeze(0).to(device, dtype=torch.bfloat16)
    print("image_tensor.shape",image_tensor.shape)

    # full_prompt = DEFAULT_IMAGE_TOKEN + "\n" + prompt_text
    full_prompt=prompt_text
    input_ids = tokenizer_image_token(full_prompt, lens_model.tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(device)
    print("input_ids",input_ids)
    print("input_ids.shape",input_ids.shape)

    inputs_embeds = lens_model.get_multimodal_inputs_embeds(input_ids, image_tensor)
    print("inputs_embeds",inputs_embeds)
    print("inputs_embeds.shape",inputs_embeds.shape)
    
    seq_len = inputs_embeds.shape[1]
    layers = sorted(set(list(lens.source_layers) + [lens_model.n_layers - 1]))
    n_layers = len(layers)

    # 2. Record Activations & Compute Layer Logits
    with ActivationRecorder(lens_model.layers, at=layers) as recorder:
        lens_model.forward(inputs_embeds)
        activations = {layer: recorder.activations[layer].detach() for layer in layers}

    top_ids = np.zeros((seq_len, n_layers, top_n), dtype=np.int32)
    top_ranks = np.zeros((seq_len, n_layers, top_n), dtype=np.int32)
    tracked_ids_set = set()

    for l_idx, layer in enumerate(layers):
        res = activations[layer][0].float()
        if layer in lens.jacobians:
            res = lens.transport(res, layer)
        logits = lens_model.unembed(res).float()
        
        top_k = torch.topk(logits, k=top_n, dim=-1)
        top_ids[:, l_idx, :] = top_k.indices.cpu().numpy()
        top_ranks[:, l_idx, :] = np.arange(top_n)[None, :]
        
        for tid in top_k.indices.flatten().tolist():
            tracked_ids_set.add(tid)

    tracked_token_ids = sorted(tracked_ids_set)
    rank_tensor = np.zeros((seq_len, n_layers, len(tracked_token_ids)), dtype=np.int32)
    
    for l_idx, layer in enumerate(layers):
        res = activations[layer][0].float()
        if layer in lens.jacobians:
            res = lens.transport(res, layer)
        logits = lens_model.unembed(res).float()
        target_tensors = torch.tensor(tracked_token_ids, device=logits.device)
        ranks = _ranks_of(logits, target_tensors)
        rank_tensor[:, l_idx, :] = ranks.cpu().numpy()

    # 3. Build SliceData
    context_token_ids = input_ids[0].tolist()                                                                                                                                                                     
    context_token_strs = []            
    if IMAGE_TOKEN_INDEX in context_token_ids:                                                                                                                                                                
        img_pos = context_token_ids.index(IMAGE_TOKEN_INDEX)                                                                                                                                                  
    else:                                                                                                                                                                                                     
        img_pos = 1  # Default fallback if not found
    n_img_tokens = seq_len - (len(context_token_ids) - 1)                                                                                                                                                     
    img_strs = [f"<img_{i}>" for i in range(max(1, n_img_tokens))]                                                                                                                                            
    img_ids = [IMAGE_TOKEN_INDEX] * len(img_strs)         
    prefix_ids = context_token_ids[:img_pos]                                                                                                                                                                  
    suffix_ids = context_token_ids[img_pos + 1:]     
    prefix_strs = []                                                                                                                                                                                          
    for t in prefix_ids:                                                                                                                                                                                      
        try:                                                                                                                                                                                                  
            prefix_strs.append(tokenizer.decode([t], clean_up_tokenization_spaces=False))                                                                                                                     
        except Exception:                                                                                                                                                                                     
            prefix_strs.append(f"token_{t}")                                                                                                                                                                  
                                                                                                                                                                                                                
    suffix_strs = []                                                                                                                                                                                          
    for t in suffix_ids:                                                                                                                                                                                      
        try:                                                                                                                                                                                                  
            suffix_strs.append(tokenizer.decode([t], clean_up_tokenization_spaces=False))                                                                                                                     
        except Exception:                                                                                                                                                                                     
            suffix_strs.append(f"token_{t}")                                                                                                                                                                  
                                                                                                                                                                                                                
    # Combine prefix + expanded image tokens + suffix                                                                                                                                                         
    context_token_strs = prefix_strs + img_strs + suffix_strs                                                                                                                                                 
    context_token_ids = prefix_ids + img_ids + suffix_ids
    print('context_token_strs',context_token_strs)                                                                                                            
    # for t in context_token_ids:                                                                                                                                                                                   
    #     if t < 0:                                                                                                                                                                                                 
    #         context_token_strs.append("<image>")                                                                                                                                                                  
    #     else:                                                                                                                                                                                                     
    #         try:                                                                                                                                                                                                  
    #             context_token_strs.append(tokenizer.decode([t], clean_up_tokenization_spaces=False))                                                                                                              
    #         except Exception:                                                                                                                                                                                     
    #             context_token_strs.append(f"token_{t}")    
    # context_token_ids = input_ids[0].tolist()
    # context_token_strs = [tokenizer.decode([t], clean_up_tokenization_spaces=False) for t in context_token_ids]
    # while len(context_token_strs) < seq_len:
    #     context_token_strs.insert(1, "[IMG]")
    #     context_token_ids.insert(1, IMAGE_TOKEN_INDEX)
        
    # vocab_fragment = {}
    # for tid in tracked_token_ids:
    #     try:
    #         vocab_fragment[tid] = tokenizer.decode([tid])
    #     except Exception:
    #         vocab_fragment[tid] = f"token_{tid}"

    vocab_fragment = {}                                                                                                                                                                                           
    for tid in tracked_token_ids:                                                                                                                                                                                 
        if tid < 0:                                                                                                                                                                                               
            vocab_fragment[tid] = "<image>"                                                                                                                                                                       
        else:                                                                                                                                                                                                     
            try:                                                                                                                                                                                                  
                vocab_fragment[tid] = tokenizer.decode([tid])                                                                                                                                                     
            except Exception:                                                                                                                                                                                     
                vocab_fragment[tid] = f"token_{tid}"

    slice_data = SliceData(
        seq_len=seq_len,
        layers=layers,
        context_token_ids=context_token_ids[:seq_len],
        context_token_strs=context_token_strs[:seq_len],
        top_ids=top_ids,
        top_ranks=top_ranks,
        tracked_token_ids=tracked_token_ids,
        rank_tensor=rank_tensor,
        vocab_fragment=vocab_fragment,
        vocab_size=getattr(tokenizer, 'vocab_size', 32000)
    )

    # 4. Render & Write HTML via jlens.vis.build_page
    out_dir = os.path.dirname(os.path.abspath(output_html_path))
    os.makedirs(out_dir, exist_ok=True)

    page_html, raw_bytes, payload_bytes = build_page(
        slice_data,
        prompt=prompt_text,
        title=f"TokenPacker Jacobian Lens: {prompt_text[:30]}",
        description=f"Multimodal Jacobian Lens readout for prompt: '{prompt_text}'",
        mode=mode,
        out_dir=out_dir if mode == "fetch" else None
    )

    with open(output_html_path, "w", encoding="utf-8") as f:
        f.write(page_html)

    print(f"Exported standalone HTML visualization to: {output_html_path}")
    print(f"Payload size: {payload_bytes / 1024:.1f} KB. Download this file to your local machine and double-click to open in any browser!")
    return page_html

print("Standalone HTML exporter function ready.")

Standalone HTML exporter function ready.


In [9]:
prompt = "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\nWhat is the person holding?\nAnswer the question using a single word or phrase. ASSISTANT: Wii remote"

In [10]:
tmp = export_multimodal_slice_html(
    lens_model,
    fitted_lens_path='/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/jacobian-lens-ATA/out/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-05242026/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202_multimodal_vqav2_lens.pt',
    image_path="/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker/playground/data/eval/vqav2/test2015/COCO_test2015_000000017515.jpg",
    prompt_text=prompt,
    output_html_path = "out/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202/visualizations/multimodal_slice.html",
    mode = "embed", 
    top_n = 10
)

Loaded Jacobian Lens from /mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/jacobian-lens-ATA/out/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-05242026/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202_multimodal_vqav2_lens.pt
image_tensor.shape torch.Size([1, 3, 336, 336])
input_ids tensor([[    1,   319, 13563,  1546,   263, 12758,  1404,   322,   385, 23116,
         21082, 20255, 29889,   450, 20255,  4076,  8444, 29892, 13173, 29892,
           322,  1248,   568,  6089,   304,   278,  1404, 29915, 29879,  5155,
         29889,  3148,  1001, 29901, 29871,  -200, 29871,    13,  5618,   338,
           278,  2022, 13587, 29973,    13, 22550,   278,  1139,   773,   263,
          2323,  1734,   470, 16549, 29889,   319,  1799,  9047, 13566, 29901,
           399,  2236,  7592]], device='cuda:0')
input_ids.shape torch.Size([1, 63])
inputs_embeds tensor([[[ 0.0029, -0.0039,  0.0020,  ..., -0.0091,  

## Section 7: Standalone 12x12 Spatial Image Grid HTML Visualization Exporter

This cell imports `export_multimodal_slice_html_with_grid`:
- Converts the exact preprocessed 336x336 `image_tensor` into a Base64 image.
- Renders an interactive **12x12 Spatial Image Grid Panel** under the heatmap.
- **Hovering over `<img_k>`** in the heatmap highlights the exact 2D patch on the preprocessed 336x336 image!

In [11]:
import os
import io
import html
import json
import base64
import gzip
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from typing import Optional, Any

from jlens.hooks import ActivationRecorder
from jlens.lens import JacobianLens
from jlens.vis import SliceData, _ranks_of, build_page


def build_patch_boxes(masks_by_scale: dict, image_px_size: int) -> list:
    """Maps a projector's per-scale selection masks to per-token pixel-fraction boxes.

    masks_by_scale: {patch_size_units: (H_p, W_p) bool/float array, batch dim already
    stripped}, e.g. {1: (24,24), 2: (12,12), 4: (6,6)}. patch_size_units is expressed
    in units of the finest scale's grid cell (raw_grid_size = the largest mask's side).

    Returns patch_boxes[k] for k = 0..N-1, ordered to match the projector's actual
    output token order: scale ascending (finest first), then row-major (row, col)
    within each scale's mask. Each box has normalized [0,1] x0/y0/x1/y1 fractions of
    the image, plus the source scale/row/col for display.
    """
    if not masks_by_scale:
        raise ValueError("masks_by_scale is empty")

    raw_grid_size = max(m.shape[-1] for m in masks_by_scale.values())
    unit_px = image_px_size // raw_grid_size
    if unit_px * raw_grid_size != image_px_size:
        raise ValueError(
            f"image_px_size={image_px_size} not evenly divisible by raw_grid_size={raw_grid_size}"
        )

    boxes = []
    for patch_size_units in sorted(masks_by_scale.keys()):
        mask = np.asarray(masks_by_scale[patch_size_units]).astype(bool)
        rows, cols = np.nonzero(mask)  # row-major traversal
        for r, c in zip(rows.tolist(), cols.tolist()):
            x0 = c * patch_size_units * unit_px
            y0 = r * patch_size_units * unit_px
            side = patch_size_units * unit_px
            boxes.append({
                "x0": x0 / image_px_size,
                "y0": y0 / image_px_size,
                "x1": (x0 + side) / image_px_size,
                "y1": (y0 + side) / image_px_size,
                "scale": patch_size_units,
                "row": r,
                "col": c,
            })
    return boxes


@torch.no_grad()
def export_multimodal_slice_html_adaptive(
    lens_model: Any,
    fitted_lens_path: str,
    image_path: str,
    prompt_text: str,
    output_html_path: str = "out/visualizations/multimodal_adaptive_slice.html",
    top_n: int = 10,
    allow_token_count_mismatch: bool = False,
):
    """Exports a standalone HTML visualization with a variable-size adaptive-scale
    spatial image overlay under the heatmap, built from the projector's own real
    per-image patch selection masks (not a uniform grid).
    """
    device = lens_model.input_device
    tokenizer = lens_model.tokenizer

    if not os.path.exists(fitted_lens_path):
        print(f"Error: Lens checkpoint {fitted_lens_path} not found.")
        return None

    lens = JacobianLens.from_pretrained(os.path.dirname(fitted_lens_path), filename=os.path.basename(fitted_lens_path))
    print(f"Loaded Jacobian Lens from {fitted_lens_path}")

    # 1. Process Image into Preprocessed Tensor
    from llava.mm_utils import tokenizer_image_token, process_images
    from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN

    raw_image = Image.open(image_path).convert('RGB')
    image_tensor = process_images([raw_image], lens_model.image_processor, lens_model.model.config)[0].unsqueeze(0).to(device, dtype=torch.bfloat16)
    image_px_size = image_tensor.shape[-1]

    # Un-normalize image_tensor [3, H, W] -> HxW PIL Image
    mean = getattr(lens_model.image_processor, 'image_mean', [0.48145466, 0.4578275, 0.40821073])
    std = getattr(lens_model.image_processor, 'image_std', [0.26862954, 0.26130258, 0.27577711])

    img_np = image_tensor.detach().squeeze(0).cpu().float().numpy()
    for c in range(3):
        img_np[c] = img_np[c] * std[c] + mean[c]
    img_np = np.clip(img_np * 255.0, 0, 255).astype(np.uint8)
    processed_pil = Image.fromarray(img_np.transpose(1, 2, 0))

    # Base64 encode the preprocessed image
    buffered = io.BytesIO()
    processed_pil.save(buffered, format="JPEG")
    image_b64 = "data:image/jpeg;base64," + base64.b64encode(buffered.getvalue()).decode()

    # 2. Compute Fused Multimodal Input Embeddings, capturing the projector's own
    # per-image selection masks immediately afterward (tmp_masks is a module-level
    # global only valid right after this forward pass).
    full_prompt = prompt_text
    input_ids = tokenizer_image_token(full_prompt, lens_model.tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(device)

    inputs_embeds = lens_model.get_multimodal_inputs_embeds(input_ids, image_tensor)
    seq_len = inputs_embeds.shape[1]

    import llava.model.multimodal_projector.builder as builder
    raw_masks = getattr(builder, "tmp_masks", None)
    if not raw_masks:
        raise RuntimeError(
            "builder.tmp_masks is empty/missing after get_multimodal_inputs_embeds(); "
            "this checkpoint's projector did not run PatchTokenizer_v2.forward() — "
            "is this a cross_attn_adaptive_v3 checkpoint? For TokenPacker-style "
            "checkpoints, use jlens.vis_multimodal.export_multimodal_slice_html_with_grid instead."
        )
    masks_by_scale = {}
    for patch_size_units, v in raw_masks.items():
        if v.shape[0] != 1:
            raise NotImplementedError(
                f"batch size {v.shape[0]} != 1 for scale {patch_size_units}; "
                "this exporter only supports single-image export (B=1)"
            )
        masks_by_scale[int(patch_size_units)] = v[0].detach().to("cpu").numpy()

    layers = sorted(set(list(lens.source_layers) + [lens_model.n_layers - 1]))
    n_layers = len(layers)

    # 3. Record Activations & Compute Layer Logits
    with ActivationRecorder(lens_model.layers, at=layers) as recorder:
        lens_model.forward(inputs_embeds)
        activations = {layer: recorder.activations[layer].detach() for layer in layers}

    top_ids = np.zeros((seq_len, n_layers, top_n), dtype=np.int32)
    top_ranks = np.zeros((seq_len, n_layers, top_n), dtype=np.int32)
    tracked_ids_set = set()

    for l_idx, layer in enumerate(layers):
        res = activations[layer][0].float()
        if layer in lens.jacobians:
            res = lens.transport(res, layer)
        logits = lens_model.unembed(res).float()

        top_k = torch.topk(logits, k=top_n, dim=-1)
        top_ids[:, l_idx, :] = top_k.indices.cpu().numpy()
        top_ranks[:, l_idx, :] = np.arange(top_n)[None, :]

        for tid in top_k.indices.flatten().tolist():
            tracked_ids_set.add(tid)

    tracked_token_ids = sorted(tracked_ids_set)
    rank_tensor = np.zeros((seq_len, n_layers, len(tracked_token_ids)), dtype=np.int32)

    for l_idx, layer in enumerate(layers):
        res = activations[layer][0].float()
        if layer in lens.jacobians:
            res = lens.transport(res, layer)
        logits = lens_model.unembed(res).float()
        target_tensors = torch.tensor(tracked_token_ids, device=logits.device)
        ranks = _ranks_of(logits, target_tensors)
        rank_tensor[:, l_idx, :] = ranks.cpu().numpy()

    # 4. Build Dynamic Token Sequence & <image> Alignment
    context_token_ids = input_ids[0].tolist()
    if IMAGE_TOKEN_INDEX in context_token_ids:
        img_pos = context_token_ids.index(IMAGE_TOKEN_INDEX)
    else:
        img_pos = 1

    n_img_tokens = seq_len - (len(context_token_ids) - 1)
    img_strs = [f"<img_{i}>" for i in range(max(1, n_img_tokens))]
    img_ids = [IMAGE_TOKEN_INDEX] * len(img_strs)

    prefix_ids = context_token_ids[:img_pos]
    suffix_ids = context_token_ids[img_pos + 1:]

    prefix_strs = [tokenizer.decode([t], clean_up_tokenization_spaces=False) if t >= 0 else "<image>" for t in prefix_ids]
    suffix_strs = [tokenizer.decode([t], clean_up_tokenization_spaces=False) if t >= 0 else "<image>" for t in suffix_ids]

    full_ctx_strs = prefix_strs + img_strs + suffix_strs
    full_ctx_ids = prefix_ids + img_ids + suffix_ids

    patch_boxes = build_patch_boxes(masks_by_scale, image_px_size)
    if len(patch_boxes) != n_img_tokens:
        msg = (
            f"token count mismatch: n_img_tokens={n_img_tokens} (from sequence length) != "
            f"len(patch_boxes)={len(patch_boxes)} (from tmp_masks). Per-scale counts: "
            f"{ {k: int(v.sum()) for k, v in masks_by_scale.items()} }"
        )
        if not allow_token_count_mismatch:
            raise RuntimeError(msg + " Pass allow_token_count_mismatch=True to export anyway.")
        print("WARNING: " + msg)

    vocab_fragment = {}
    for tid in tracked_token_ids:
        if tid < 0 or tid == IMAGE_TOKEN_INDEX:
            vocab_fragment[tid] = "<image>"
        else:
            try:
                vocab_fragment[tid] = tokenizer.decode([tid])
            except Exception:
                vocab_fragment[tid] = f"token_{tid}"

    slice_data = SliceData(
        seq_len=seq_len,
        layers=layers,
        context_token_ids=full_ctx_ids[:seq_len],
        context_token_strs=full_ctx_strs[:seq_len],
        top_ids=top_ids,
        top_ranks=top_ranks,
        tracked_token_ids=tracked_token_ids,
        rank_tensor=rank_tensor,
        vocab_fragment=vocab_fragment,
        vocab_size=getattr(tokenizer, 'vocab_size', 32000)
    )

    # Render HTML page using slice_vis_multimodal_adaptive.html (standalone new template)
    out_dir = os.path.dirname(os.path.abspath(output_html_path))
    os.makedirs(out_dir, exist_ok=True)

    from importlib.resources import files
    from jlens.vis import _slice_meta, _slice_bin, _template

    meta = _slice_meta(slice_data, prompt_text, f"Adaptive Multi-Scale Lens: {prompt_text[:30]}", f"Multimodal adaptive-patch spatial readout for prompt: '{prompt_text}'", None, None)
    meta["image_b64"] = image_b64
    meta["patch_boxes"] = patch_boxes
    meta["img_start"] = img_pos
    meta["n_img_tokens"] = n_img_tokens

    ranks = slice_data.rank_tensor.astype("<i4")
    file_map = {"slice.bin": _slice_bin(slice_data)} | {
        f"ranks/{tid}.bin": gzip.compress(ranks[:, :, i].tobytes(), compresslevel=6)
        for i, tid in enumerate(slice_data.tracked_token_ids)
    }

    bootstrap = {
        "mode": "embed",
        "meta": meta,
        "files": {
            name: base64.b64encode(body).decode() for name, body in file_map.items()
        },
    }

    bootstrap_json = json.dumps(bootstrap, ensure_ascii=False).replace("</", "<\\/")

    try:
        template_str = (files("jlens") / "data" / "slice_vis_multimodal_adaptive.html").read_text(encoding="utf-8")
    except Exception:
        import jlens
        template_str = (Path(jlens.__file__).parent / "data" / "slice_vis_multimodal_adaptive.html").read_text(encoding="utf-8")

    d3_tag = _template("embed").split("<style>")[0]

    page_html = (
        template_str
        .replace("__TITLE__", html.escape(f"Adaptive Multi-Scale Lens: {prompt_text[:30]}"))
        .replace("__WHAT__", html.escape(f"Multimodal adaptive-patch spatial readout for prompt: '{prompt_text}'"))
        .replace("__D3__", d3_tag)
        .replace("__BOOTSTRAP__", bootstrap_json)
    )

    with open(output_html_path, "w", encoding="utf-8") as f:
        f.write(page_html)

    payload_bytes = sum(len(b) for b in file_map.values())
    print(f"🎉 Exported Adaptive Multi-Scale Spatial HTML to: {output_html_path}")
    print(f"Payload size: {payload_bytes / 1024:.1f} KB. Download to your machine and open in any browser!")
    return page_html


In [12]:
prompt = "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\nWhat is the person holding?\nAnswer the question using a single word or phrase. ASSISTANT: Wii remote"

In [14]:
tmp = export_multimodal_slice_html_adaptive(
    lens_model,
    fitted_lens_path='/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/jacobian-lens-ATA/out/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-05242026/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202_multimodal_vqav2_lens.pt',
    image_path="/mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/TokenPacker/playground/data/eval/vqav2/test2015/COCO_test2015_000000017515.jpg",
    prompt_text=prompt,
    output_html_path = "out/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202/visualizations/multimodal_slice_grid.html",
    # mode = "embed", 
    top_n = 10
)

Loaded Jacobian Lens from /mnt/pvc-shared-pvc-data-volume-ea328235/MLLM/jacobian-lens-ATA/out/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-05242026/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202_multimodal_vqav2_lens.pt
🎉 Exported Adaptive Multi-Scale Spatial HTML to: out/llava-cross_attn_adaptive-it-thresholdpred-30-multilev-dubconv-llava_v1_5_mix665k-en-h100-0524202/visualizations/multimodal_slice_grid.html
Payload size: 59114.9 KB. Download to your machine and open in any browser!
